# DEMO: QUẢN LÝ MỐC NẠP DỮ LIỆU (WATERMARK) & MERGE VÀO VALUE VÀ HISTORY

Notebook này giải quyết 2 bài toán lớn trong Data Engineering:
1. **Xác định mốc nạp dữ liệu (Watermark Control Pattern)**: Làm thế nào để từng đợt chạy tự động biết cần quét dữ liệu từ mốc thời gian nào đến mốc thời gian nào, không bị sót và không bị nạp thừa.
2. **Xử lý đồng bộ 2 bảng**:
   - **Bảng Value (`customer_value`)**: Snapshot 1:1 giá trị hiện tại (SCD Type 1).
   - **Bảng History (`customer_history`)**: Lưu vết lịch sử biến động có `valid_from`, `valid_to`, `is_current` (SCD Type 2/Temporal).
3. **Cập nhật Watermark**: Chỉ cập nhật mốc mới khi toàn bộ quá trình nạp thành công.

## BƯỚC 1: KHỞI TẠO SPARK VÀ BẢNG QUẢN LÝ MỐC NẠP (CTL_WATERMARK_CONTROL)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, current_timestamp, lit, expr, max as spark_max
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

# Khởi tạo Spark Session
spark = SparkSession.builder \
    .appName("CDC_Watermark_Merge_Value_History") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# 1. Khởi tạo Bảng Quản lý Mốc Thời Gian (Control Watermark Table)
# Dùng datetime.now() thay vì current_timestamp() trong Python tuple
init_watermark_data = [
    ('customers', datetime(2026, 1, 1, 8, 0), datetime(2026, 1, 1, 8, 0), 'SUCCESS', datetime.now())
]
ctl_schema = ['table_name', 'last_success_ts', 'current_cutoff_ts', 'status', 'updated_at']
df_ctl = spark.createDataFrame(init_watermark_data, ctl_schema)
df_ctl.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('ctl_watermark_control_v2')

print('--- BẢNG QUẢN LÝ MỐC NẠP (WATERMARK CONTROL) ---')
spark.table('ctl_watermark_control_v2').show(truncate=False)


## BƯỚC 2: KHỞI TẠO BẢNG VALUE VÀ BẢNG HISTORY BAN ĐẦU (TẠI MỐC 08:00)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, BooleanType

# 1. Bảng Value ban đầu (Khởi tạo mốc 08:00, có cột _ingest_at phục vụ Incremental Load lên Gold)
val_data = [
    ('CUST_101', 'Nguyen Van An', 'Hanoi', 15000000.0, datetime(2026, 1, 1, 8, 0), datetime(2026, 1, 1, 8, 0)),
    ('CUST_102', 'Tran Thi Binh', 'Danang', 20000000.0, datetime(2026, 1, 1, 8, 0), datetime(2026, 1, 1, 8, 0))
]

val_schema = StructType([
    StructField('customer_id', StringType(), False),
    StructField('customer_name', StringType(), True),
    StructField('address', StringType(), True),
    StructField('credit_limit', DoubleType(), True),
    StructField('updated_at', TimestampType(), True),     # Thời gian cập nhật nghiệp vụ từ nguồn
    StructField('_ingest_at', TimestampType(), False)     # Thời gian ghi vào Lakehouse (dùng làm Watermark cho Gold)
])

spark.createDataFrame(val_data, val_schema) \
    .write.format('delta').mode('overwrite').option('overwriteSchema', 'true') \
    .saveAsTable('customer_value_v2')

# 2. Bảng History ban đầu (Định nghĩa StructType tường minh cho cột valid_to chứa giá trị None)
hist_data = [
    ('CUST_101', 'Nguyen Van An', 'Hanoi', 15000000.0, datetime(2026, 1, 1, 8, 0), None, True, datetime(2026, 1, 1, 8, 0)),
    ('CUST_102', 'Tran Thi Binh', 'Danang', 20000000.0, datetime(2026, 1, 1, 8, 0), None, True, datetime(2026, 1, 1, 8, 0))
]

hist_schema = StructType([
    StructField('customer_id', StringType(), False),
    StructField('customer_name', StringType(), True),
    StructField('address', StringType(), True),
    StructField('credit_limit', DoubleType(), True),
    StructField('valid_from', TimestampType(), False),
    StructField('valid_to', TimestampType(), True),       # Khai báo rõ TimestampType dù giá trị ban đầu là None
    StructField('is_current', BooleanType(), False),
    StructField('_inserted_at', TimestampType(), False)
])

spark.createDataFrame(hist_data, hist_schema) \
    .write.format('delta').mode('overwrite').option('overwriteSchema', 'true') \
    .saveAsTable('customer_history_v2')

print('--- BẢNG VALUE BAN ĐẦU (CÓ CỘT _ingest_at ĐẦY ĐỦ) ---')
spark.table('customer_value_v2').show(truncate=False)
print('--- BẢNG HISTORY BAN ĐẦU ---')
spark.table('customer_history_v2').show(truncate=False)


## BƯỚC 3: MÔ PHỎNG DỮ LIỆU BRONZE CDC (CHỨA NHIỀU MỐC THỜI GIAN KHÁC NHAU)

Giả sử trong Bronze CDC có cả dữ liệu cũ (lúc 07:00, 08:00) và dữ liệu mới phát sinh (từ 08:30 đến 10:30).

In [ ]:
bronze_data = [
    # Dữ liệu cũ đã nạp từ trước (<= 08:00) -> Job PHẢI BỎ QUA
    ('CUST_101', 'Nguyen Van An', 'Hanoi', 15000000.0, datetime(2026, 1, 1, 8, 0), 'I', 1001),
    
    # Dữ liệu MỚI phát sinh trong đợt này (> 08:00 và <= 10:30):
    # 1. CUST_101 update hạn mức lúc 09:00
    ('CUST_101', 'Nguyen Van An', 'Hanoi', 18000000.0, datetime(2026, 1, 1, 9, 0), 'U', 1002),
    # 2. CUST_101 tiếp tục update địa chỉ lúc 10:00
    ('CUST_101', 'Nguyen Van An', 'Haiphong', 18000000.0, datetime(2026, 1, 1, 10, 0), 'U', 1003),
    # 3. CUST_102 bị xóa tài khoản lúc 10:15
    ('CUST_102', 'Tran Thi Binh', 'Danang', 20000000.0, datetime(2026, 1, 1, 10, 15), 'D', 1004),
    # 4. CUST_103 tạo mới lúc 10:30
    ('CUST_103', 'Pham Dang Khoa', 'Cantho', 25000000.0, datetime(2026, 1, 1, 10, 30), 'I', 1005)
]
bronze_schema = ['customer_id', 'customer_name', 'address', 'credit_limit', 'cdc_timestamp', 'cdc_op', 'cdc_scn']
df_bronze_all = spark.createDataFrame(bronze_data, bronze_schema)
df_bronze_all.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('bronze_customers_cdc_v2')

print('--- TOÀN BỘ DỮ LIỆU TRONG BRONZE CDC (CHƯA LỌC) ---')
spark.table('bronze_customers_cdc_v2').show()

## BƯỚC 4: XÁC ĐỊNH MỐC NẠP THEO THỜI ĐIỂM BẮT ĐẦU JOB (WINDOW FREEZE PATTERN)

**Nguyên tắc chuẩn Enterprise:**
1. `start_watermark`: Mốc chạy thành công của lần trước (lấy từ `ctl_watermark_control_v2`).
2. `cutoff_watermark`: **Cố định tại thời điểm bắt đầu đợt chạy này (`batch_start_time`)**, KHÔNG lấy `max(cdc_timestamp)` để tránh mốc chặn bị trôi khi Kafka đổ dữ liệu liên tục.
3. Mọi dữ liệu phát sinh sau mốc `cutoff_watermark` sẽ được nhường lại cho đợt chạy kế tiếp.

In [ ]:
# 1. Lấy mốc bắt đầu từ lần chạy thành công trước
ctl_row = spark.table('ctl_watermark_control_v2').filter("table_name = 'customers'").first()
start_watermark = ctl_row['last_success_ts']

# 2. CHỐT CHẶN MỐC KẾT THÚC = THỜI ĐIỂM BẮT ĐẦU CHẠY ĐỢT NÀY (Window Freeze)
# Giả sử đợt chạy này kích hoạt lúc 10:30:00 (hoặc nhận từ Airflow data_interval_end)
batch_start_time = datetime(2026, 1, 1, 10, 30, 0)
cutoff_watermark = batch_start_time

print(f"==> MỐC BẮT ĐẦU (start_watermark):         {start_watermark}")
print(f"==> MỐC CHỐT CHẶN ĐỢT NÀY (cutoff_watermark): {cutoff_watermark}")

# 3. LỌC CHÍNH XÁC KHOẢNG THỜI GIAN: (start_watermark < cdc_timestamp <= cutoff_watermark)
df_incremental_batch = spark.table('bronze_customers_cdc_v2').filter(
    (col('cdc_timestamp') > start_watermark) & (col('cdc_timestamp') <= cutoff_watermark)
)

print('--- TẬP DỮ LIỆU INCREMENTAL ĐƯỢC CHỐT ĐỂ XỬ LÝ TRONG PHIÊN NÀY ---')
df_incremental_batch.show()


## BƯỚC 5: XỬ LÝ MERGE VÀO BẢNG VALUE (SCD TYPE 1)
- Bảng Value lưu giữ trạng thái mới nhất của khách hàng.
- **Cột `_ingest_at`**: Được gán bằng `current_timestamp()` mỗi khi INSERT hoặc UPDATE. Đây chính là cột mốc Watermark để luồng ETL từ Silver lên Gold (Star Schema/ClickHouse) trích xuất dữ liệu tăng dần.

In [ ]:
# Khử trùng lặp trong đợt này (lấy bản ghi mới nhất cho mỗi customer_id)
win_latest = Window.partitionBy('customer_id').orderBy(col('cdc_timestamp').desc(), col('cdc_scn').desc())
df_value_latest = df_incremental_batch \
    .withColumn('rn', row_number().over(win_latest)) \
    .filter(col('rn') == 1) \
    .drop('rn')

tbl_value = DeltaTable.forName(spark, 'customer_value_v2')

# MERGE vào Bảng Value (Cập nhật _ingest_at = current_timestamp() cho cả Update và Insert)
tbl_value.alias('tgt').merge(
    df_value_latest.alias('src'),
    'tgt.customer_id = src.customer_id'
).whenMatchedDelete(
    condition = "src.cdc_op = 'D'"
).whenMatchedUpdate(
    condition = "src.cdc_op = 'U'",
    set = {
        'customer_name': 'src.customer_name',
        'address': 'src.address',
        'credit_limit': 'src.credit_limit',
        'updated_at': 'src.cdc_timestamp',
        '_ingest_at': 'current_timestamp()'
    }
).whenNotMatchedInsert(
    condition = "src.cdc_op = 'I'",
    values = {
        'customer_id': 'src.customer_id',
        'customer_name': 'src.customer_name',
        'address': 'src.address',
        'credit_limit': 'src.credit_limit',
        'updated_at': 'src.cdc_timestamp',
        '_ingest_at': 'current_timestamp()'
    }
).execute()

print('--- BẢNG VALUE SAU KHI NẠP INCREMENTAL (CÓ _ingest_at MỚI NHẤT) ---')
spark.table('customer_value_v2').show(truncate=False)


## BƯỚC 6: XỬ LÝ ĐỒNG BỘ VÀO BẢNG HISTORY (XỬ LÝ ĐA SỰ KIỆN TRONG 1 BATCH - SCD TYPE 2)

**Bí quyết xử lý khi 1 khách hàng có NHIỀU lần Update trong cùng 1 Batch:**
1. **Đóng bản ghi cũ đang mở**: Chỉ lấy sự kiện thay đổi ĐẦU TIÊN của khách hàng đó trong batch để đóng bản ghi cũ (tránh lỗi `DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE`).
2. **Xâu chuỗi các sự kiện bên trong batch**: Dùng hàm `lead(cdc_timestamp)` để tự động nối tiếp các khoảng `valid_from` và `valid_to` giữa các lần update trong cùng 1 batch.

In [ ]:
tbl_history = DeltaTable.forName(spark, 'customer_history_v2')

# 1. ĐÓNG BẢN GHI ĐANG MỞ CŨ TRONG BẢNG HISTORY
# Lưu ý: Chỉ lấy sự kiện đầu tiên trong batch của mỗi khách hàng để merge đóng bản ghi cũ
win_first_change = Window.partitionBy('customer_id').orderBy(col('cdc_timestamp').asc(), col('cdc_scn').asc())
df_first_change = df_incremental_batch.filter("cdc_op IN ('U', 'D')") \
    .withColumn('rn', row_number().over(win_first_change)) \
    .filter(col('rn') == 1) \
    .drop('rn')

tbl_history.alias('hist').merge(
    df_first_change.alias('cdc'),
    'hist.customer_id = cdc.customer_id AND hist.is_current = TRUE'
).whenMatchedUpdate(
    condition = 'cdc.cdc_timestamp > hist.valid_from',
    set = {
        'valid_to': 'cdc.cdc_timestamp',
        'is_current': expr('false')
    }
).execute()

# 2. XÂU CHUỖI VÀ CHÈN CÁC PHIÊN BẢN MỚI TRONG BATCH (DÙNG LEAD ĐỂ NỐI DÒNG THỜI GIAN)
win_lead = Window.partitionBy('customer_id').orderBy(col('cdc_timestamp').asc(), col('cdc_scn').asc())
df_batch_timeline = df_incremental_batch.filter("cdc_op != 'D'") \
    .withColumn('valid_from', col('cdc_timestamp')) \
    .withColumn('valid_to', expr('lead(cdc_timestamp) over (partition by customer_id order by cdc_timestamp asc, cdc_scn asc)')) \
    .withColumn('is_current', expr('valid_to IS NULL')) \
    .withColumn('_inserted_at', current_timestamp()) \
    .select('customer_id', 'customer_name', 'address', 'credit_limit', 'valid_from', 'valid_to', 'is_current', '_inserted_at')

# Chèn các phiên bản mới vào bảng History (dùng MERGE chống trùng khi replay)
tbl_history.alias('tgt').merge(
    df_batch_timeline.alias('src'),
    'tgt.customer_id = src.customer_id AND tgt.valid_from = src.valid_from'
).whenNotMatchedInsertAll().execute()

print('--- BẢNG HISTORY V2 SAU KHI CẬP NHẬT SCD TYPE 2 HOÀN HẢO ---')
spark.table('customer_history_v2').orderBy('customer_id', 'valid_from').show(truncate=False)


## BƯỚC 7: CẬP NHẬT WATERMARK MỚI SAU KHI PIPELINE THÀNH CÔNG HOÀN TOÀN

In [ ]:
# Khi cả Bảng Value và Bảng History đã commit thành công:
# Cập nhật last_success_ts = cutoff_watermark để đợt chạy tiếp theo sẽ bắt đầu từ mốc 10:30!

spark.sql(f"""
    UPDATE ctl_watermark_control_v2
    SET last_success_ts = cast('{cutoff_watermark}' as timestamp),
        current_cutoff_ts = cast('{cutoff_watermark}' as timestamp),
        status = 'SUCCESS',
        updated_at = current_timestamp()
    WHERE table_name = 'customers'
""")

print('--- BẢNG WATERMARK CONTROL ĐÃ ĐƯỢC CẬP NHẬT MỐC MỚI ---')
spark.table('ctl_watermark_control_v2').show(truncate=False)
print('===> Đợt chạy tiếp theo sẽ tự động chỉ đọc dữ liệu phát sinh sau mốc 10:30:00!')